# Bioinvasion analysis: demo

Runs the analysis of Rohrbach et al. (2025) with observations of *Megachile sculpturalis* (the species of the paper) downloaded from iNaturalist, on the demo cost surface of this repository. The same functions are used by the command line (`cli.py`) and the web app.

The demo cost surface is based on terrain only (GTOPO30 elevation, costs 1 to 10), not on the habitat suitability model of the paper, so the results differ from the published ones. They show how the analysis works, not how the species spread.

The settings are written to `demo_project/project.toml` next to this notebook; all results go into that folder. Every stage is skipped if its inputs and settings have not changed, so re-running cells is cheap.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))  # repository root

from IPython.display import Image, Markdown, display
from bioinvasion import pipeline
from bioinvasion.config import load_config
from bioinvasion.report import insights

SETTINGS = '''
[data]
source = "inaturalist"
taxon = "Megachile sculpturalis"
research_grade_only = true
include_obscured = false
max_positional_accuracy_m = 5000

[cost]
file = "../../data/cost_surface_gtopo30_esri102031_5km_exp_base1_k9.tif"
label = "terrain5km"
'''
os.makedirs('demo_project', exist_ok=True)
with open('demo_project/project.toml', 'w', encoding='utf-8') as f:
    f.write(SETTINGS)
cfg = load_config('demo_project')

## 1. Observations
Downloads all research-grade observations within the extent of the cost surface (about one request per second, so this can take a few minutes the first time). To download again later, e.g. for new observations, use `force=True`.

In [ ]:
pipeline.import_presence(cfg)
print(open('demo_project/presence/import_report.txt', encoding='utf-8').read())

## 2. Least-cost paths
Removes observations the analysis cannot use, keeps the earliest observation per cost surface cell and connects each observation to the nearest earlier one.

In [ ]:
paths_dir = pipeline.compute_paths(cfg)
pipeline.load_paths(paths_dir)['manifest']['results']

## 3. Sensitivity test
An overview over the whole range of accumulated costs, then a high-resolution run over the suggested range. The rule behind the suggestion is printed, followed by the stable ranges of the high-resolution run (widest first).

In [ ]:
coarse, fine, chosen = pipeline.sensitivity_test(cfg)
display(Markdown(chosen['rule']))
d = insights.nearest_definition(fine, cfg.delineation.robust_definition)
ranges = insights.stable_ranges(fine, d)
ranges

## 4. Populations and expansion rates
**Choose the threshold yourself**, after looking at the sensitivity test (in your own projects, put it into `[delineation] threshold` of `project.toml`). For this demo, the beginning of the widest stable range is used.

In [ ]:
threshold = ranges[0]['from']
out_dir, summary, checks = pipeline.report(cfg, threshold=threshold)
print(summary)
display(Markdown('\n'.join(f'- {text}' for level, text in checks)))

In [ ]:
for name in sorted(os.listdir(out_dir)):
    if name.endswith('.png'):
        display(Image(os.path.join(out_dir, name), width=800))